# 06-1. 損失関数 — 動かして確かめる

📖 解説: [`../01_loss_functions.md`](../01_loss_functions.md)

## このノートで触るもの
1. MSE (回帰)
2. クロスエントロピー (分類)
3. ソフトマックス + クロスエントロピー
4. JAX で線形回帰の訓練
5. JAX でロジスティック回帰の訓練
6. L2 正則化の効果

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (01_loss_functions.md)](../01_loss_functions.md)

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)

%matplotlib inline
rng = np.random.default_rng(42)

## 1. MSE (Mean Squared Error)

$$
\text{MSE} = \frac{1}{N} \sum_i (y_i - \hat{y}_i)^2
$$

対応するコード: `jnp.mean((y_true - y_pred) ** 2)`

In [ ]:
def mse(y_true, y_pred):
    return jnp.mean((y_true - y_pred) ** 2)

y_true = jnp.array([1.0, 2.0, 3.0, 4.0])
y_pred = jnp.array([1.1, 1.9, 3.2, 3.8])
print(f'MSE = {float(mse(y_true, y_pred)):.6f}')
print(f'手計算: ((0.1)² + (0.1)² + (0.2)² + (0.2)²)/4 = {(0.01 + 0.01 + 0.04 + 0.04)/4:.6f}')

## 2. クロスエントロピー

$$
\text{CE} = -\sum_k y_k \log \hat{y}_k
$$

対応するコード: `-jnp.sum(y_true * jnp.log(y_pred + ε))`

In [ ]:
EPSILON = 1e-12

def cross_entropy(y_true, y_pred):
    """y_true: ワンホット (N, K), y_pred: ソフトマックス出力 (N, K)."""
    return -jnp.mean(jnp.sum(y_true * jnp.log(y_pred + EPSILON), axis=1))

# 3 クラス分類
y_true = jnp.array([
    [1, 0, 0],   # サンプル1: 真値クラス0
    [0, 1, 0],   # サンプル2: 真値クラス1
])

# 良い予測 vs 悪い予測
y_pred_good = jnp.array([[0.95, 0.04, 0.01], [0.03, 0.94, 0.03]])
y_pred_bad  = jnp.array([[0.10, 0.85, 0.05], [0.80, 0.10, 0.10]])

print(f'良い予測 → CE = {float(cross_entropy(y_true.astype(float), y_pred_good)):.6f}')
print(f'悪い予測 → CE = {float(cross_entropy(y_true.astype(float), y_pred_bad)):.6f}')

## 3. ソフトマックス + クロスエントロピー

$$
\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

数値安定化のため `log_softmax` を使う:

$$
\log p_i = z_i - \log \sum_j e^{z_j}
$$

In [ ]:
def cross_entropy_logits(logits, labels):
    """数値安定な softmax + CE.
    
    Args:
        logits: (N, K) — softmax 前のスコア
        labels: (N,)   — クラスインデックス
    """
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_probs[jnp.arange(len(labels)), labels])

logits = jnp.array([[2.0, 1.0, 0.1],
                    [0.5, 2.5, 0.3]])
labels = jnp.array([0, 1])
print(f'CE (logits版) = {float(cross_entropy_logits(logits, labels)):.6f}')

# Optax バージョン
y_onehot = jax.nn.one_hot(labels, num_classes=3)
ce_optax = optax.softmax_cross_entropy(logits, y_onehot)
print(f'CE (Optax)   = {float(jnp.mean(ce_optax)):.6f}')

## 4. 線形回帰の訓練 (MSE で)

モデル: $\hat{y} = w x + b$, 損失: $\frac{1}{N}\sum (y - \hat{y})^2$

勾配: $\frac{\partial L}{\partial w} = -\frac{2}{N}\sum x(y - wx - b)$

In [ ]:
# データ生成 y = 2x + 1 + ノイズ
X = jnp.linspace(-5, 5, 100)
y = 2 * X + 1 + 0.5 * jnp.array(rng.normal(0, 1, 100))

def loss(params, x, y):
    w, b = params
    return jnp.mean((y - (w * x + b))**2)

# 訓練
params = (jnp.array(0.0), jnp.array(0.0))
lr = 0.01
history = []

for epoch in range(100):
    val, grads = jax.value_and_grad(loss)(params, X, y)
    params = (params[0] - lr * grads[0], params[1] - lr * grads[1])
    history.append(float(val))

print(f'真値: w=2.0, b=1.0')
print(f'学習後: w={float(params[0]):.4f}, b={float(params[1]):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE')
axes[0].set_yscale('log')
axes[0].set_title('損失の推移')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X, y, alpha=0.3, label='データ')
x_line = jnp.linspace(-5, 5, 10)
axes[1].plot(x_line, params[0] * x_line + params[1], 'r-', linewidth=2, label=f'予測: y = {float(params[0]):.2f}x + {float(params[1]):.2f}')
axes[1].set_title('線形回帰の結果')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. ロジスティック回帰 (二値分類, BCE で)

モデル: $\hat{y} = \sigma(wx + b)$, 損失: $-[y\log\hat{y} + (1-y)\log(1-\hat{y})]$

In [ ]:
# データ生成: x が大きいとクラス1
n = 200
X = jnp.array(rng.normal(0, 2, n))
y = (X > 0).astype(jnp.float32)  # 真の決定境界は x = 0

def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def bce_loss(params, x, y):
    w, b = params
    p = sigmoid(w * x + b)
    return -jnp.mean(y * jnp.log(p + 1e-12) + (1 - y) * jnp.log(1 - p + 1e-12))

params = (jnp.array(0.5), jnp.array(0.0))
for epoch in range(500):
    grads = jax.grad(bce_loss)(params, X, y)
    params = (params[0] - 0.1 * grads[0], params[1] - 0.1 * grads[1])

print(f'学習後: w={float(params[0]):.4f}, b={float(params[1]):.4f}')
print(f'最終 BCE 損失: {float(bce_loss(params, X, y)):.6f}')

# 可視化
x_grid = jnp.linspace(-6, 6, 100)
p_grid = sigmoid(params[0] * x_grid + params[1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X, y, alpha=0.4, s=15, label='データ (0 or 1)')
ax.plot(x_grid, p_grid, 'r-', linewidth=2, label='予測確率 σ(wx+b)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('x'); ax.set_ylabel('y / 確率')
ax.set_title('ロジスティック回帰')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. L2 正則化の効果

$$
L_{\text{total}} = L_{\text{data}} + \lambda \|\boldsymbol{\theta}\|^2
$$

→ 大きい重みを罰して過学習を防ぐ

In [ ]:
# 多項式回帰 (次数 10 で過学習しやすい)
n = 30
X = jnp.linspace(-1, 1, n)
y_true = jnp.sin(2 * X)
y = y_true + 0.2 * jnp.array(rng.normal(0, 1, n))

# 特徴量: [1, x, x², ..., x^10]
def poly_features(x, degree=10):
    return jnp.stack([x**i for i in range(degree + 1)], axis=-1)

Phi = poly_features(X)

def loss_with_reg(theta, Phi, y, lam):
    pred = Phi @ theta
    mse_loss = jnp.mean((y - pred)**2)
    reg = lam * jnp.sum(theta**2)
    return mse_loss + reg

# 💡 ポイント: 1 ステップ更新を @jax.jit でコンパイル
# こうしないと Python ループから毎回 jax.grad を呼ぶことになり超遅い (約 250x の差)
grad_fn = jax.grad(loss_with_reg)

@jax.jit
def step(theta, lam, lr=0.005):
    return theta - lr * grad_fn(theta, Phi, y, lam)

def fit(lam, n_iter=10_000):
    theta = jnp.zeros(11)
    for _ in range(n_iter):
        theta = step(theta, lam)
    return theta

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_dense = jnp.linspace(-1.2, 1.2, 200)
Phi_dense = poly_features(x_dense)

for ax, lam in zip(axes, [0.0, 0.001, 0.1]):
    theta = fit(lam)
    pred = Phi_dense @ theta
    ax.scatter(X, y, alpha=0.5, label='データ')
    ax.plot(x_dense, jnp.sin(2 * x_dense), 'g--', alpha=0.7, label='真の関数')
    ax.plot(x_dense, pred, 'r-', linewidth=2, label=f'予測 (λ={lam})')
    ax.set_title(f'λ = {lam}')
    ax.set_ylim(-2, 2)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('→ λ = 0 では過学習、λ大きすぎでは過小適合、適切な λ で良いバランス')

## まとめ
- 回帰には MSE、分類にはクロスエントロピー
- ソフトマックス + log は数値安定化のため `log_softmax` を使う
- 線形回帰・ロジスティック回帰は JAX で 10 行で訓練できる
- L2 正則化で過学習を防ぐ

## 次へ
→ [`02_backprop.ipynb`](02_backprop.ipynb)  解説 → [`../02_backprop.md`](../02_backprop.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [章 TOP](../README.md) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`02_backprop.ipynb`](02_backprop.ipynb) |